# Segmento 1: Il limite del loop naive

Abbiamo un vector database con 1.000 ricette indicizzate dalla sessione scorsa.

Proviamo a usarlo in una conversazione: l'utente chiede una ricetta, poi ne chiede un'altra.

Funzionerà?

In [16]:
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
import os

load_dotenv()
client = OpenAI()
qdrant = QdrantClient(host="localhost", port=6333)

EMBED_MODEL = "text-embedding-3-small"
COLLECTION = "recipes"

# Verifichiamo che Qdrant abbia i dati della sessione 2
if not qdrant.collection_exists(COLLECTION):
    print(f"La collection '{COLLECTION}' non esiste!")
    print("Vai nel notebook session2/segment_2.ipynb ed eseguilo per crearla.")
else:
    info = qdrant.get_collection(COLLECTION)
    print(f"Collection '{COLLECTION}': {info.points_count} ricette indicizzate")

Collection 'recipes': 1000 ricette indicizzate


In [17]:
def search_recipes(query, top_n=5):
    """Cerca ricette in Qdrant per similarità semantica."""
    response = client.embeddings.create(input=[query], model=EMBED_MODEL)
    query_vector = response.data[0].embedding

    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        limit=top_n,
        with_payload=True,
    )

    for point in results.points:
        print(f"  {point.score:.4f}  {point.payload['title']}")

## Simuliamo una conversazione

Turno 1: l'utente cerca una ricetta con pomodoro.

In [22]:
# Turno 1: una query chiara e specifica
print('Utente: "ricetta con pomodoro"\n')
search_recipes("ricetta con il pesce")

Utente: "ricetta con pomodoro"

  0.4823  Whole Branzino Roasted in Salt
  0.4739  Crispy Fish with Brown Butter Sauce and Kohlrabi Salad
  0.4699  Crispy Za'atar Fish With Israeli Couscous, Swiss Chard, and Feta
  0.4655  Black Sea Bass with Moroccan Vegetables and Chile Sauce
  0.4639  Monkfish with Ratatouille


Funziona! Risultati pertinenti.

Ora l'utente ne vuole un'altra, come farebbe in una conversazione naturale.

In [23]:
# Turno 2: una richiesta che dipende dal contesto
print('Utente: "Give me another recipe"\n')
search_recipes("Dammene un'altra")

Utente: "Give me another recipe"

  0.2819  Pappa al Pomodoro
  0.2702  Fresh Pasta with Crabmeat, Peas and Chile
  0.2574  Penne with Tomato Pesto and Smoked Mozzarella
  0.2541  Linguine with Burst Tomatoes and Chiles
  0.2537  Mozzarella Arrabiata Salsa


## Cosa è successo?

La frase "dammene un'altra" è stata embeddata **alla lettera**, senza nessun contesto.

Il vector database non sa che stavamo parlando di ricette con pomodoro. Ogni query è **indipendente**.

Per questo, il retrieval viene fuorviato da caratteristiche secondarie della query, come la lingua.

Per risolvere questo problema ci serve qualcosa che:
1. **Mantenga il contesto** della conversazione
2. **Decida da solo** quando e come cercare nel database

Questo qualcosa è il **function calling**: torniamo alle slides.